# Document Ingestion Testing

In [1]:
# packages

## link project directory
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

## custom packages
from ingestion.documents.sec import SecEdgarProvider
from ingestion.documents.parser import DocumentParser

## data collection
import sqlite3
import pandas as pd

## other
from tqdm import tqdm

In [2]:
# constants
from constants import (
    DEFAULT_SQLITE_PATH,
    USER_AGENT, FILING_TYPES, START_DATE, END_DATE
)

In [3]:
# connect to data
connection = sqlite3.connect(DEFAULT_SQLITE_PATH)
connection.row_factory = sqlite3.Row

# retrieve CIKs
query = "select distinct company_id, cik from companies"
ciks = pd.read_sql_query(query, connection).set_index('company_id').to_dict()['cik']


## Retrieve Raw HTML

In [4]:
provider = SecEdgarProvider(user_agent =  USER_AGENT)
filings = provider.list_filings(
    cik = ciks['GOOG'], 
    filing_types = FILING_TYPES, 
    start_date = START_DATE, end_date = END_DATE
)
for filing in filings:
    print(
        filing.filing_type,
        filing.filing_date,
        filing.accession_number
    )

10-Q 2026-04-30 0001652044-26-000048
10-K 2026-02-05 0001652044-26-000018
10-Q 2025-10-30 0001652044-25-000091
10-Q 2025-07-24 0001652044-25-000062
10-Q 2025-04-25 0001652044-25-000043
10-K 2025-02-05 0001652044-25-000014
10-Q 2024-10-30 0001652044-24-000118
10-Q 2024-07-24 0001652044-24-000079
10-Q 2024-04-26 0001652044-24-000053
10-K 2024-01-31 0001652044-24-000022
10-Q 2023-10-25 0001652044-23-000094
10-Q 2023-07-26 0001652044-23-000070


In [5]:
# iterate over the CIKs and get all filings

# define the provider
provider = SecEdgarProvider(user_agent =  USER_AGENT)

# iterate through each company to get the filings
# ciks = {'GOOG': <cik>, 'AAPL': <cik>, ...}
seen = set() # track seen CIKs to avoid duplicates (i.e., GOOG, GOOGL)
# for ticker, cik in tqdm(
#     ciks.items(),
#     desc = 'Companies',
#     unit = 'company'
# ):
for ticker, cik in ciks.items():
    # check if we have already seen this CIK (i.e., GOOG, GOOGL)
    if cik in seen:
        continue
    seen.add(cik)

    # get the filings
    filings = provider.list_filings(
        cik = cik,
        filing_types = FILING_TYPES,
        start_date = START_DATE,
        end_date = END_DATE
    )

    # store each of the retrieved HTML documents
    for f in tqdm(filings, desc = f'[{ticker}] Downloading filings', unit = 'filing'):
        # download the filing (or understand why failure occurred)
        try:
            html_doc = provider.download_filing(f)
        except Exception as e:
            print(f'Failed: {ticker} {f.accession_number}: {e}')
            continue
        
        # store the document in the database

        ## define the filepath
        fdate = f.filing_date.strftime("%Y-%m-%d")
        fname = f'{fdate}_{f.filing_type}_{f.accession_number}'
        fpath = PROJECT_ROOT / 'data' / 'raw' / 'sec_edgar' / ticker / f'{fname}.html'
        if fpath.exists():
            continue # skip if the file already exists

        ## write the file
        fpath.parent.mkdir(parents = True, exist_ok = True)
        with open(fpath, 'w', encoding='utf-8') as f:
            f.write(html_doc)

[TSLA] Downloading filings: 100%|██████████| 26/26 [00:01<00:00, 25.26filing/s]


In [6]:
# load test file
from bs4 import BeautifulSoup

test_path = (
    PROJECT_ROOT / 'data' / 'raw' / 'sec_edgar' / 'GOOG' / 
    '2023-07-26_10-Q_0001652044-23-000070.html'
)

with open(test_path, 'r', encoding = 'utf-8') as f:
    soup = BeautifulSoup(f, 'html.parser')

In [7]:
# get testing filing metadata
provider = SecEdgarProvider(user_agent =  USER_AGENT)
test_filing = provider.list_filings(
    cik = ciks['GOOG'],
    filing_types = FILING_TYPES,
    start_date = '2023-07-01',
    end_date = '2023-08-01'
)[0]

In [8]:
test_filing

FilingMetadata(cik='0001652044', accession_number='0001652044-23-000070', filing_type='10-Q', filing_date=datetime.date(2023, 7, 26), primary_document='goog-20230630.htm', filing_url='https://www.sec.gov/Archives/edgar/data/1652044/000165204423000070/goog-20230630.htm')

In [9]:
dp = DocumentParser()
dp.load_html(test_path)
dp.clean_dom()
dp.extract_metadata(
    filing_metadata = test_filing,
    ticker = 'GOOG'
)
dp.remove_boilerplate()
dp.extract_sections()

# print(f'{dp.__class__.__name__} Attributes')
# for k, v in vars(dp.metadata).items():
#     print(f'\t{k}: {v}')

# print(dp.soup.body.get_text("\n", strip=True)[:5000])
# print(dp.soup.prettify())

# NOTE: this returns 1664 evals of Parts / Items but only 9 section / parts
dp.sections

[{'part': 'Part I',
  'item': 'ITEM 1',
  'title': 'FINANCIAL STATEMENTS',
  'element': <div style="margin-top:3pt;padding-left:72pt;text-align:justify;text-indent:-72pt"><span style="color:#000000;font-family:'Arial',sans-serif;font-size:10pt;font-weight:700;line-height:120%">ITEM 1.</span><span style="color:#000000;font-family:'Arial',sans-serif;font-size:10pt;font-weight:700;line-height:120%;padding-left:37.04pt">FINANCIAL STATEMENTS</span></div>},
 {'part': 'Part I',
  'item': 'ITEM 2',
  'title': 'MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS',
  'element': <div style="margin-top:14pt;padding-left:72pt;text-align:justify;text-indent:-72pt"><span style="color:#000000;font-family:'Arial',sans-serif;font-size:10pt;font-weight:700;line-height:120%">ITEM 2.</span><span style="color:#000000;font-family:'Arial',sans-serif;font-size:10pt;font-weight:700;line-height:120%;padding-left:37.04pt">MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITIO

In [10]:
# with open('cleaned.html', 'w', encoding = 'utf-8') as f:
#     f.write(str(dp.soup))